In [1]:
!pip install google-cloud-bigquery pyarrow

In [2]:
API_URL = "https://api.coingecko.com/api/v3/coins/markets"

PARAMS = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 20,
    "page": 1,
    "sparkline": False
}

In [3]:
import requests
import logging

logging.basicConfig(level=logging.INFO)

def fetch_crypto_data():
    try:
        logging.info("Fetching data from CoinGecko API...")

        response = requests.get(API_URL, params=PARAMS)

        response.raise_for_status()

        data = response.json()

        logging.info("Data fetched successfully.")

        return data

    except requests.exceptions.RequestException as e:
        logging.error(f"API request failed: {e}")
        return None

data = fetch_crypto_data()

print(data[:2])

[{'id': 'bitcoin', 'symbol': 'btc', 'name': 'Bitcoin', 'image': 'https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400', 'current_price': 75833, 'market_cap': 1520220840885, 'market_cap_rank': 1, 'fully_diluted_valuation': 1520220840885, 'total_volume': 36296251255, 'high_24h': 77881, 'low_24h': 75220, 'price_change_24h': -868.4172799733205, 'price_change_percentage_24h': -1.13221, 'market_cap_change_24h': -15079651127.08081, 'market_cap_change_percentage_24h': -0.9822, 'circulating_supply': 20035015.0, 'total_supply': 20035015.0, 'max_supply': 21000000.0, 'ath': 126080, 'ath_change_percentage': -39.85332, 'ath_date': '2025-10-06T18:57:42.558Z', 'atl': 67.81, 'atl_change_percentage': 111733.03214, 'atl_date': '2013-07-06T00:00:00.000Z', 'roi': None, 'last_updated': '2026-05-27T08:12:12.998Z'}, {'id': 'ethereum', 'symbol': 'eth', 'name': 'Ethereum', 'image': 'https://coin-images.coingecko.com/coins/images/279/large/ethereum.png?1696501628', 'current_price': 2083.

In [4]:
import pandas as pd

# Convert JSON to DataFrame
df = pd.DataFrame(data)

# Select important columns
df = df[[
    'id',
    'symbol',
    'name',
    'current_price',
    'market_cap',
    'total_volume',
    'price_change_percentage_24h',
    'market_cap_rank',
    'last_updated'
]]

# Rename columns
df.rename(columns={
    'id': 'coin_id',
    'symbol': 'coin_symbol',
    'name': 'coin_name'
}, inplace=True)

# Handle null values
df.fillna(0, inplace=True)

# Create derived field
df['market_strength'] = df['market_cap'] / df['total_volume']

# Convert last_updated to datetime
df['last_updated'] = pd.to_datetime(df['last_updated'])

# Display transformed data
print(df.head())

       coin_id coin_symbol coin_name  current_price     market_cap  \
0      bitcoin         btc   Bitcoin   75833.000000  1520220840885   
1     ethereum         eth  Ethereum    2083.180000   251456822624   
2       tether        usdt    Tether       0.998603   189315923812   
3  binancecoin         bnb       BNB     652.170000    87963523891   
4       ripple         xrp       XRP       1.330000    82264173566   

   total_volume  price_change_percentage_24h  market_cap_rank  \
0   36296251255                     -1.13221                1   
1   14562149726                     -0.62126                2   
2   64754524268                     -0.01545                3   
3    1068695951                     -0.72120                4   
4    1572998600                     -0.82974                5   

                      last_updated  market_strength  
0 2026-05-27 08:12:12.998000+00:00        41.883687  
1 2026-05-27 08:12:12.945000+00:00        17.267837  
2 2026-05-27 08:12:12.0900

In [5]:
df.fillna(0, inplace=True)

In [9]:
df['market_strength'] = df['market_cap'] / df['total_volume']
df.head()

,coin_id,coin_symbol,coin_name,current_price,market_cap,total_volume,price_change_percentage_24h,market_cap_rank,last_updated,market_strength
0,bitcoin,btc,Bitcoin,75833.000000,1520220840885,36296251255,-1.13221,1,2026-05-27 08:12:12.998000+00:00,41.883687
1,ethereum,eth,Ethereum,2083.180000,251456822624,14562149726,-0.62126,2,2026-05-27 08:12:12.945000+00:00,17.267837
2,tether,usdt,Tether,0.998603,189315923812,64754524268,-0.01545,3,2026-05-27 08:12:12.090000+00:00,2.923594
3,binancecoin,bnb,BNB,652.170000,87963523891,1068695951,-0.72120,4,2026-05-27 08:12:13.201000+00:00,82.309214
4,ripple,xrp,XRP,1.330000,82264173566,1572998600,-0.82974,5,2026-05-27 08:12:13.188000+00:00,52.297678
